# DeepSCF: from molecules to a periodic polymer
This tutorial reproduces the **polyethylene example** from [*Convolutional network learning of self-consistent electron density via grid-projected atomic fingerprints*](https://doi.org/10.1038/s41524-024-01433-0) paper, using a Google Colab tutorial setup.

In the following steps, we learn how to prepare the input dataset and use the pretrained **(R+S)×3 model** to predict the electron density. We then compare the resulting total energy, atomic forces, and band structure with those obtained from the initial atomic density and a converged self-consistent field (SCF) calculation.

## 1. Setup
Use a **CPU runtime** and run the cells in order.

### Clone the DeepSCF repositories
Download the fixed DeepSCF snapshot and this tutorial into `/content`. DeepSCF provides the model and modified SIESTA source; `tutorial/deepscf/01/src` contains the polymer inputs and Python helpers.

The next cell checks the snapshot and required files. Existing clone directories are reused.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d DeepSCF ]; then
  git clone https://github.com/yhkimlab/DeepSCF.git
  git -C DeepSCF checkout --detach fe3c3c18052b51101e438dad2ffa06f94e77b5bb
fi

if [ ! -d tutorial ]; then
  git clone --depth 1 --branch main https://github.com/ryonggyulee/tutorial.git tutorial
fi


In [ ]:
%%bash
set -e
cd /content/DeepSCF
# Check the tutorial snapshot.
[ "$(git rev-parse HEAD)" = fe3c3c18052b51101e438dad2ffa06f94e77b5bb ]
test -s requirements.txt
test -s src/siesta-4.1-b4/Obj/gfortran-system.make
git log -1 --format='%h %cs %s'
test -s /content/tutorial/deepscf/01/src/inputs/scf/RUN.fdf
test -s /content/tutorial/deepscf/01/src/inputs/nscf/RUN.fdf
test -s /content/tutorial/deepscf/01/src/predict_polymer.py
echo "DeepSCF and tutorial inputs ready"


### Install Python libraries
Install the Python packages, then import them and check their versions.

In [ ]:
%pip install -q torch h5py hydra-core==1.3.2 omegaconf==2.3.0 "sisl[viz]" matplotlib

In [ ]:
%matplotlib inline
import sys, subprocess
import torch, h5py, numpy, hydra, omegaconf
import sisl, sisl.viz

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| sisl:', sisl.__version__)

### Build the custum SIESTA code
Compile the modified SIESTA bundled with DeepSCF. This version writes the additional grids needed for preprocessing and reads the predicted density for the non-SCF calculation. Compilation may take a few minutes; the full log is saved to `/content/siesta-build.log`.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential binutils gfortran libblas-dev liblapack-dev
cd /content/DeepSCF/src/siesta-4.1-b4/Obj
sh ../Src/obj_setup.sh
cp gfortran-system.make arch.make
# Save the build log.
make -j2 > /content/siesta-build.log 2>&1 || { tail -80 /content/siesta-build.log; exit 1; }
test -x siesta
echo "SIESTA build OK"


**Check the executable.** Launch SIESTA without an input file and check the version and compiler information. The missing-input error and nonzero exit code are expected here; this cell only checks that the executable starts.

In [ ]:
import subprocess, tempfile

with tempfile.TemporaryDirectory() as workdir:
    result = subprocess.run(
        ['/content/DeepSCF/src/siesta-4.1-b4/Obj/siesta'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
        cwd=workdir, timeout=15,
    )
print(result.stdout)
print(result.stderr)
print('Exit code:', result.returncode, '(no input supplied)')


## 2. Reference SCF calculation
### Prepare the run folders
Copy the supplied inputs into separate folders for reference SCF, initial-density non-SCF, and ML-density non-SCF calculations. Each run uses a new working directory, printed below, so previous output files are not reused.

To repeat the calculation, start again from this cell and run the subsequent cells.

In [ ]:
from pathlib import Path
import sys, shutil, tempfile

ROOT = Path('/content/DeepSCF')
SRC = Path('/content/tutorial/deepscf/01/src')
PYTHON = sys.executable
SIESTA = ROOT / 'src/siesta-4.1-b4/Obj/siesta'
(ROOT / 'results').mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='polymer-', dir=ROOT / 'results'))
SCF, NSCF = WORK / 'scf/polymer', WORK / 'nscf'
INITIAL = WORK / 'initial'
for folder, mode in [(SCF, 'scf'), (INITIAL, 'nscf'), (NSCF, 'nscf')]:
    shutil.copytree(SRC / 'inputs' / mode, folder)
print('Working directory:', WORK)

### Check the polymer structure
Read the supplied `STRUCT.fdf` with sisl. It contains **24 atoms (C₈H₁₆)** in a **20 × 20 × 10 Å** cell. The polymer extends periodically along **z**; the transverse directions contain vacuum.

The figure is an **x–z projection**, so atoms separated along y may overlap. Check the chain and the cell boundaries before running the calculation.

In [ ]:
geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
print('Number of atoms:', len(geometry))
print('Cell vectors (Angstrom):\n', geometry.cell)

geometry.plot(
    axes='xyz',
    show_cell='box',
    backend='plotly'
).show()

### Check the FDF inputs
The `src/inputs/scf/` and `src/inputs/nscf/` folders each contain a complete input set: `RUN.fdf`, `STRUCT.fdf`, `C.psf`, and `H.psf`. Each `RUN.fdf` contains all calculation settings and includes only the atomic structure from `STRUCT.fdf`. Routine default options are omitted.

The full reference `RUN.fdf` and structure are displayed below.

| Input setting | Purpose in this example |
|---|---|
| `PAO.BasisSize DZP`, `XC.authors PBE` | Use a double-zeta polarized basis and PBE. |
| `kgrid_Monkhorst_Pack` | Sample the periodic chain with a 1 × 1 × 8 k-point mesh. |
| `MeshSizes 80 80 40` | Fix the grid at 0.25 Å spacing for the pretrained model; no `MeshCutoff` is specified. |
| `MD.NumCGsteps 0` | Keep the supplied atomic positions fixed. |
| `WriteForces T` | Print atomic forces; this SIESTA version also writes `Molecular.FA` automatically. |
| `SaveRho`, `SaveIonicCharge` | Write density grids used in preprocessing. |
| `SaveDeltaRho T` | Write `Molecular.DRHO` in both SCF and NSCF runs. |
| `MaxSCFIterations 300`, `SCF.MustConverge T` | Require the reference SCF calculation to converge. |

`BandLines` specifies $\Gamma \rightarrow X$, where $X = (0, 0, \tfrac{1}{2})$ in reciprocal-lattice coordinates along the polymer axis. The bands are folded into the supplied 10 Å repeat cell.

In [ ]:
from IPython.display import HTML, display
from html import escape

def show_input(path):
    # Render the full file without a scrolling output area.
    display(HTML(f'<h4>{escape(path.name)}</h4>'
                 f'<pre style="max-height: none; overflow: visible; white-space: pre-wrap;">'
                 f'{escape(path.read_text())}</pre>'))

for name in ['RUN.fdf', 'STRUCT.fdf']:
    show_input(SCF / name)


### Run the reference SCF calculation
Run SIESTA with the checked inputs. The helper checks successful completion and SCF convergence. The `iscf` header and `scf:` iteration rows appear live below; the full output is also saved to `stdout.txt` and `stderr.txt` in the SCF folder. The helper disables gfortran stdout buffering and flushes each progress line.

The converged density `Molecular.RHO` and bands `Molecular.bands` will be used as references. The modified SIESTA also writes the grids consumed by DeepSCF preprocessing.

In [ ]:
subprocess.run([PYTHON, '-u', str(SRC / 'run_siesta.py'), str(SCF),
                '--siesta', str(SIESTA)], check=True)

### Check the reference density
Compare the initial atomic density with the converged SCF density. The plots show the density integrated along y, in e/Å², rather than a single plane through the structure.

The reported relative error uses the **full 3D grid**:
\[
\epsilon_\rho = 100\,\frac{\sum_i |\rho_i-\rho_i^{\mathrm{SCF}}|}{\sum_i \rho_i^{\mathrm{SCF}}}.
\]
The uniform grid volume cancels in this ratio. The integrated SCF charge should be close to **48 valence electrons** for C₈H₁₆.

In [ ]:
sys.path.insert(0, str(SRC))
from analysis import plot_density, plot_bands, compare_observables

plot_density(initial=SCF / 'rhoatom', scf=SCF / 'Molecular.RHO');

### Check the deformation density
The deformation density is $\Delta\rho = \rho_{\mathrm{SCF}} - \rho_{\mathrm{atoms}}$, relative to the superposition of neutral atomic densities. Electron accumulation between atoms can reveal covalent bonding.

Use sisl and Plotly to rotate the 3D isosurfaces: orange shows electron accumulation ($\Delta\rho > 0$), and blue shows depletion ($\Delta\rho < 0$). Adjust `isovalue` to explore the redistribution.

`SaveDeltaRho T` in the reference `RUN.fdf` enables this output. If you ran the calculation before adding this option, rerun the input preparation and reference SCF cells first.


In [ ]:
drho_path = SCF / 'Molecular.DRHO'
if not drho_path.exists():
    raise FileNotFoundError(
        f'{drho_path} is missing. Rerun the reference SCF with SaveDeltaRho T.'
    )
dr_geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
dr = sisl.get_sile(str(drho_path)).read_grid(geometry=dr_geometry)

# Positive deformation density highlights electron accumulation in covalent bonds.
# Choose equal positive/negative levels within the range of both signs.
isovalue = 0.2 * min(float(dr.grid.max()), -float(dr.grid.min()))
if isovalue <= 0:
    raise ValueError('Expected both accumulation and depletion in the deformation density.')
dr.plot(
    axes='xyz',
    isos=[
        {'val': isovalue, 'color': 'orange', 'opacity': 0.6, 'name': 'Electron accumulation'},
        {'val': -isovalue, 'color': 'royalblue', 'opacity': 0.6, 'name': 'Electron depletion'},
    ],
    plot_geom=True,
    show_cell='box',
    backend='plotly',
).show()


### Check the reference bands
Plot the bands near the gap along $\Gamma$–$X$. The energy zero is the **reference SCF valence-band maximum** on this path.

The printed gap is obtained from the sampled $\Gamma$–$X$ eigenvalues. This is a path-based comparison, not a separate search over the full Brillouin zone.

In [ ]:
plot_bands(scf=SCF);

## 3. Prepare the model inputs
### Convert: SIESTA grids → HDF5
Preprocessing consists of three stages: **convert → determine scaling → collect**. We run conversion first, check the pretrained model's scaling parameters, and then collect the normalized inputs.

`1.convert.py` reads the four SIESTA grids and writes one HDF5 file per channel. Each file contains a single dataset named `value`, with shape `(80, 80, 40)` corresponding to the x, y, and z grid dimensions.

| SIESTA file | Output under `raw/` | Contents |
|---|---|---|
| `Molecular.RHO` | `target/polymer.h5` | Converged SCF electron density |
| `Molecular.IOCH` | `feature1/polymer.h5` | Ionic charge density |
| `rhoatom` | `feature2/polymer.h5` | Initial atomic electron density |
| `rhoover` | `feature3/polymer.h5` | Orbital-overlap feature |

This step changes the file format without normalizing or resampling the grid values. The arrays retain the numerical units of the SIESTA grids; the cell and atomic coordinates are not stored in these HDF5 files. Check the dataset names, shapes, and value ranges below.


In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/1.convert.py'),
                str(WORK / 'scf'), str(WORK / 'raw')], check=True)

channels = ['target', 'feature1', 'feature2', 'feature3']
for channel in channels:
    path = WORK / 'raw' / channel / 'polymer.h5'
    with h5py.File(path, 'r') as data:
        values = data['value'][:]
        print(f'{path.relative_to(WORK)}: keys={list(data.keys())}, '
              f'shape={values.shape}, dtype={values.dtype}, '
              f'range=[{values.min():.6g}, {values.max():.6g}]')


### Scale: check the pretrained normalization
In the full preprocessing workflow, `2.scale.py` computes the spatial mean and standard deviation of each channel for each structure, then averages these statistics over the supplied structures:

$$
\mu_c = \frac{1}{N}\sum_{n=1}^{N}\operatorname{mean}_{\mathbf r} x_c^{(n)}(\mathbf r),
\qquad
\sigma_c = \frac{1}{N}\sum_{n=1}^{N}\operatorname{std}_{\mathbf r} x_c^{(n)}(\mathbf r).
$$

Thus, `std` is the **average of the per-structure standard deviations**, rather than the standard deviation of all grid points pooled together.

Here we use the supplied `scale_r_plus_s_x3.txt` for the pretrained $(R+S)\times 3$ model. We do not run `2.scale.py` on this polymer: inference must use the scaling associated with the trained model.

| Entries in the scale file | Raw channel | Use in `3.collect.py` |
|---|---|---|
| `mean1`, `std1` | `target`: SCF density | Recorded statistics; target is copied unchanged |
| `mean2`, `std2` | `feature1`: ionic charge | Normalize input channel 0 |
| `mean3`, `std3` | `feature2`: atomic density | Normalize input channel 1 |
| `mean4`, `std4` | `feature3`: orbital overlap | Normalize input channel 2 |

Print the file below. The numbering includes the target first, so `feature1` uses **mean2/std2**, not mean1/std1.


In [ ]:
SCALE = ROOT / 'datasets/scale/scale_r_plus_s_x3.txt'
print(SCALE.read_text())

scale = {}
for line in SCALE.read_text().splitlines():
    if line.strip():
        key, value = line.split('=')
        scale[key.strip()] = float(value)


### Collect: normalize and assemble the dataset
`3.collect.py` applies a fixed affine transformation to each input channel:

$$
\widetilde{x}_c(\mathbf r) =
\frac{x_c(\mathbf r)-\mu_c}{\sigma_c}.
$$

The three normalized arrays are stacked along a new channel axis and saved together with the unmodified SCF target in `dataset/polymer.h5`.

| Dataset | Shape | Contents |
|---|---|---|
| `feature` | `(3, 80, 80, 40)` | Normalized ionic charge, atomic density, and orbital overlap, in that order |
| `target` | `(80, 80, 40)` | SCF reference density, without normalization |

Only `feature` enters the network. The prediction helper converts it to float32 and adds a batch dimension, giving `(1, 3, 80, 80, 40)`. The target is retained for error evaluation.

The cell below checks the stored arrays against the normalization formula and prints each channel's mean and standard deviation. These need not be exactly zero and one for this polymer because the scaling parameters are fixed across structures.


In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/3.collect.py'),
                str(WORK / 'raw'), str(SCALE),
                str(WORK / 'dataset')], check=True)

with h5py.File(WORK / 'dataset/polymer.h5', 'r') as dataset:
    feature = dataset['feature'][:]
    target = dataset['target'][:]
    print('HDF5 datasets:', list(dataset.keys()))
    print('feature:', feature.shape, '| target:', target.shape)

for i, channel in enumerate(channels[1:]):
    with h5py.File(WORK / 'raw' / channel / 'polymer.h5', 'r') as data:
        raw = data['value'][:]
    expected = (raw - scale[f'mean{i + 2}']) / scale[f'std{i + 2}']
    numpy.testing.assert_allclose(feature[i], expected)
    print(f'feature[{i}] ({channel}): mean={feature[i].mean():.6f}, '
          f'std={feature[i].std():.6f}')

with h5py.File(WORK / 'raw/target/polymer.h5', 'r') as data:
    numpy.testing.assert_array_equal(target, data['value'][:])
print('Verified: three normalized features and an unchanged SCF target.')


## 4. Predict the electron density
### Download the pretrained model
Download the pretrained archive and keep only `pretrained/r_plus_s_x3/model.pt`. The download is about 486 MB and is skipped if the weights are already present.

This tutorial uses inference only: neither the training dataset nor a training run is required.

In [ ]:
%%bash
set -euo pipefail
cd /content/DeepSCF
if [ ! -s pretrained/r_plus_s_x3/model.pt ]; then
  curl -fL --retry 3 https://github.com/yhkimlab/DeepSCF/releases/download/DeepSCF_v1/pretrained-models-v1.tar.gz \
    | tar --no-same-owner -xz pretrained/r_plus_s_x3/model.pt
fi
test -s pretrained/r_plus_s_x3/model.pt


### Run DeepSCF
Load the $(R+S)\times 3$ model and predict the density on CPU. The helper writes `predict/polymer.RHO` using the same cell and grid as the reference calculation and checks the exported file.

Keep this predicted density for the next step; it will be passed to SIESTA as `In.RHO`.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'predict_polymer.py'), str(ROOT), str(WORK)], check=True)
PREDICTED = WORK / 'predict/polymer.RHO'

### Compare the predicted and SCF densities
Compare the atomic, predicted, and reference densities on the same grid. The final panel shows the projected difference $\rho_{\mathrm{ML}} - \rho_{\mathrm{SCF}}$.

Use the full-grid relative errors to quantify the improvement over the atomic starting density. The difference plot helps locate the regions where the prediction differs from the reference.

In [ ]:
plot_density(initial=SCF / 'rhoatom', ml=PREDICTED, scf=SCF / 'Molecular.RHO');

### Compare the predicted and SCF deformation densities
Compare $\rho_{\mathrm{ML}} - \rho_{\mathrm{atoms}}$ (left) with $\rho_{\mathrm{SCF}} - \rho_{\mathrm{atoms}}$ (right) using the same atomic reference. Electron accumulation between atoms can reveal covalent bonding.

Both 3D panels use the same isovalue: orange marks electron accumulation and blue marks depletion. Rotate each panel to check the bonding regions; adjust `comparison_isovalue` to explore other levels. These grids are calculated from the existing density files, so no additional SIESTA run is needed.


In [ ]:
from analysis import density
from sisl.viz import merge_plots

atomic_grid = density(SCF / 'rhoatom')
ml_grid = density(PREDICTED)
scf_grid = density(SCF / 'Molecular.RHO')
comparison_geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()

# Subtract the same atomic reference on matching grids, in e/Angstrom^3.
deformation_grids = []
for grid in (ml_grid, scf_grid):
    if grid.shape != atomic_grid.shape or not numpy.allclose(grid.cell, atomic_grid.cell):
        raise ValueError('Density grids must have the same shape and cell.')
    deformation = grid.copy()
    deformation.grid = grid.grid - atomic_grid.grid
    deformation.set_geometry(comparison_geometry)
    deformation_grids.append(deformation)

# Use one absolute level for both panels and both signs.
comparison_isovalue = 0.2 * min(
    min(float(grid.grid.max()), -float(grid.grid.min()))
    for grid in deformation_grids
)
if comparison_isovalue <= 0:
    raise ValueError('Expected accumulation and depletion in both deformation densities.')
print(f'Shared isovalue: ±{comparison_isovalue:.5f} e/Angstrom^3')

plots = [
    grid.plot(
        axes='xyz',
        isos=[
            {'val': comparison_isovalue, 'color': 'orange', 'opacity': 0.6,
             'name': f'{label}: accumulation'},
            {'val': -comparison_isovalue, 'color': 'royalblue', 'opacity': 0.6,
             'name': f'{label}: depletion'},
        ],
        plot_geom=True,
        show_cell='box',
        backend='plotly',
    )
    for label, grid in zip(('DeepSCF', 'SCF'), deformation_grids)
]
merge_plots(
    *plots,
    composite_method='subplots',
    rows=1,
    cols=2,
    subplot_titles=('DeepSCF deformation density', 'SCF deformation density'),
    backend='plotly',
).show()


## 5. Non-SCF calculation
### Prepare and check the inputs
Use three cases at the **same fixed geometry**: the initial atomic density ($\rho_{\mathrm{init}}$), the predicted density ($\rho_{\mathrm{ML}}$), and the converged SCF reference ($\rho_{\mathrm{SCF}}$). Run two separate one-step calculations, using `rhoatom` and the prediction as `In.RHO`, respectively. The reference is the converged calculation from Section 2.

Both SCF and NSCF inputs set `SaveDeltaRho T` to write `Molecular.DRHO`. The helper checks that this file exists and is nonempty after each run.

Both one-step runs use the complete `src/inputs/nscf/RUN.fdf` shown below:
- `SCF.Read.Charge T` reads `In.RHO`.
- `MaxSCFIterations 1` performs one iteration.
- `SCF.Mix.First F` disables first-step mixing.
- `SCF.MustConverge F` permits completion without convergence.

The geometry, pseudopotentials, basis, grid, and k points are identical. `DM.UseSaveDM F` prevents reuse of a converged density matrix. `WriteForces T` prints the forces, and `Molecular.FA` stores atom indices and $F_x$, $F_y$, $F_z$ in eV/Å.


In [ ]:
for folder, rho in [(INITIAL, SCF / 'rhoatom'), (NSCF, PREDICTED)]:
    shutil.copy(rho, folder / 'In.RHO')
    print('Density input:', rho, '->', folder / 'In.RHO')
show_input(NSCF / 'RUN.fdf')


### Run the one-step calculations
Calculate the electronic structure separately from $\rho_{\mathrm{init}}$ and $\rho_{\mathrm{ML}}$. The helper verifies **exactly one SCF iteration** and checks that a force file was written for each run. Each run displays only one iteration (`iscf = 1`), then finishes its energy, force, and band outputs. An `SCF_NOT_CONV` message in the saved log is expected.

These are actual SIESTA calculations, not quantities supplied by the neural network. SIESTA diagonalizes the Hamiltonian constructed using the input density, then uses the output density matrix (`DM_out`) for its final energy and forces. The table therefore compares **results of calculations started from each density**, rather than direct evaluations of $E[\rho_{\mathrm{init}}]$ or $E[\rho_{\mathrm{ML}}]$.


In [ ]:
for label, folder in [('rho_init', INITIAL), ('rho_ML', NSCF)]:
    print(label)
    subprocess.run([PYTHON, '-u', str(SRC / 'run_siesta.py'), str(folder),
                    '--siesta', str(SIESTA), '--non-scf'], check=True)


### Compare total energies and atomic forces
The table compares $\rho_{\mathrm{init}}$ | $\rho_{\mathrm{ML}}$ | $\rho_{\mathrm{SCF}}$. Density errors refer to the supplied grids; energies and forces are read from the corresponding SIESTA outputs. All energy entries use the same final `siesta: Total` field.

Energy error is $\lvert E - E_{\mathrm{SCF}}\rvert/N_{\mathrm{atom}}$. Force component RMSE is $\sqrt{\sum_{i\alpha}(F_{i\alpha} - F_{i\alpha}^{\mathrm{SCF}})^2/(3N_{\mathrm{atom}})}$; maximum force-vector error is $\max_i\lVert\mathbf{F}_i - \mathbf{F}_i^{\mathrm{SCF}}\rVert$. The structure is fixed, so the SCF forces need not vanish.

Compare the initial and ML columns to assess how much the prediction improves the subsequent calculation. Small density errors alone do not guarantee accurate forces. These non-self-consistent forces are a comparison with SCF, not a demonstration of conservative forces for ML-driven relaxation or molecular dynamics.


In [ ]:
from IPython.display import Markdown, display
summary = compare_observables(initial=INITIAL, ml=NSCF, scf=SCF)
display(Markdown(summary))

### Check the atomic forces
Compare the atomic force vectors on the same fixed structure: **initial density | DeepSCF | SCF**. Red arrows show the force direction and magnitude using the same `force_scale` in all three panels. Increase this shared scale to see smaller forces more clearly.

The printed maximum force magnitudes are in eV/Å. The table above provides quantitative force errors relative to SCF; the converged SCF forces need not vanish for this fixed geometry.


In [ ]:
from sisl.viz import merge_plots

force_scale = 2.0  # Shared visual scale for all three cases.
force_plots = []
force_titles = ('Initial density', 'DeepSCF', 'SCF')
for label, directory in zip(force_titles, (INITIAL, NSCF, SCF)):
    # RUN.fdf includes the structure and SystemLabel used to find force outputs.
    fdf = sisl.get_sile(str(directory / 'RUN.fdf'))
    geometry = fdf.read_geometry()
    forces = fdf.read_force()
    if forces is None:
        raise FileNotFoundError(f'No force output found in {directory}. Run SIESTA first.')
    forces = numpy.asarray(forces)
    if forces.shape != (len(geometry), 3) or not numpy.isfinite(forces).all():
        raise ValueError(f'Invalid atomic forces for {label}: {forces.shape}')
    print(f'{label}: forces shape = {forces.shape}, '
          f'max force = {numpy.linalg.norm(forces, axis=1).max():.6f} eV/Angstrom')
    force_plots.append(geometry.plot(
        axes='xyz',
        show_cell='box',
        show_bonds=True,
        arrows={
            'data': forces,
            'name': f'{label}: atomic force',
            'color': 'red',
            'width': 4,
            'scale': force_scale,
        },
        backend='plotly',
    ))

merge_plots(
    *force_plots,
    composite_method='subplots',
    rows=1,
    cols=3,
    subplot_titles=force_titles,
    backend='plotly',
).show()


### Compare the band structures
Overlay the $\Gamma$–$X$ bands for the same three cases. All curves use the reference SCF valence-band maximum as a common energy zero.


In [ ]:
plot_bands(initial=INITIAL, ml=NSCF, scf=SCF);